<a href="https://colab.research.google.com/github/elliemeltzer/Problem-Set-2-Econ_3916_PS2/blob/main/ps2_EM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS2 — The Survey That Wasn't Random

**ECON 3916 · Applied Data Analytics** · Topics 5-8 · 100 points · your role: Data Analyst, Growth team

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is data analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. You write the simulations, the bootstrap loop and the tests yourself. AI is authorised and required in Phase 4. If you cannot explain a line you submitted, you cannot submit it.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Simulate the Population You Cannot See (30 pts)

### Step 1.1: Build a world with a known answer (15 pts)

In [2]:
# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_USERS = 50_000
BASE_BEFORE, BASE_AFTER = 3.432, 3.532      # true means 3.30 and 3.40

def simulate_population(base, n=N_USERS, rng=rng):
    """Return a DataFrame with engagement_minutes, engagement_z, had_support_ticket,
    latent s and the reported 1–5 satisfaction score.
    """
    engagement_minutes = rng.lognormal(mean=3.5, sigma=0.6, size=n)

    engagement_z = (
        (engagement_minutes - engagement_minutes.mean())
        / engagement_minutes.std()
    )

    had_support_ticket = rng.binomial(1, 0.22, size=n)

    s = (
        base
        + 0.35 * engagement_z
        - 0.60 * had_support_ticket
        + rng.normal(0, 0.75, size=n)
    )

    satisfaction = np.clip(np.rint(s), 1, 5).astype(int)

    return pd.DataFrame({
        "engagement_minutes": engagement_minutes,
        "engagement_z": engagement_z,
        "had_support_ticket": had_support_ticket,
        "s": s,
        "satisfaction": satisfaction
    })

before = simulate_population(BASE_BEFORE)
after = simulate_population(BASE_AFTER)

true_before = before["satisfaction"].mean()
true_after = after["satisfaction"].mean()
true_change = true_after - true_before

print(f"True pre-redesign mean:  {true_before:.2f}")
print(f"True post-redesign mean: {true_after:.2f}")
print(f"True improvement:        {true_change:.2f}")

True pre-redesign mean:  3.29
True post-redesign mean: 3.39
True improvement:        0.10


The simulated population has an average satisfaction score of approximately 3.30 before the redesign and 3.40 after it, so the true improvement is only about 0.10 points

### Step 1.2: Impose the response mechanism (15 pts)

In [3]:
def respond(df, a, b, rng=rng):
    """Who answers the survey: P(respond) = logistic(a + b*(s - 3.3) + 0.30*engagement_z
    - 0.25*had_support_ticket); a user answers when a uniform draw is below it."""
    z = (
        a
        + b * (df["s"] - 3.3)
        + 0.30 * df['engagement_z']
        - 0.25 * df["had_support_ticket"]
    )

    response_probability = 1 / (1 + np.exp(-z))
    responded = rng.uniform(size=len(df)) < response_probability
    result = df.copy()
    result["response_probability"] = response_probability
    result["responded"] = responded

    return result

before = respond(before, a=-1.0, b=0.45)
after = respond(after, a=-2.75, b=2.00)

before_respondents = before.loc[before["responded"]].copy()
after_respondents = after.loc[after["responded"]].copy()

observed_before = before_respondents["satisfaction"].mean()
observed_after = after_respondents["satisfaction"].mean()
observed_change = observed_after - observed_before

response_rate_before = before["responded"].mean()
response_rate_after = after["responded"].mean()

print(f"Observed pre-redesign mean:  {observed_before:.3f}")
print(f"Observed post-redesign mean: {observed_after:.3f}")
print(f"Observed change:             {observed_change:.3f}")
print(f"Pre-redesign response rate:  {response_rate_before:.1%}")
print(f"Post-redesign response rate: {response_rate_after:.1%}")
print(f"Pre respondents:             {len(before_respondents):,}")
print(f"Post respondents:            {len(after_respondents):,}")


Observed pre-redesign mean:  3.600
Observed post-redesign mean: 4.304
Observed change:             0.704
Pre-redesign response rate:  27.5%
Post-redesign response rate: 15.2%
Pre respondents:             13,768
Post respondents:            7,597


> **🛑 Verification checkpoint.** You should get an observed pre-redesign mean of
> about **3.60** (accept 3.55 to 3.65) and an observed post-redesign mean of
> about **4.31** (accept 4.25 to 4.35) — an apparent gap of roughly **0.71**
> points out of a true improvement of **0.10**. Response rates should fall from
> roughly 27% to roughly 15%. If your numbers are outside those bands, your
> response function or your `base` values are wrong. **Do not proceed until it
> matches** — everything after this depends on reproducing the PM's slide.

This is nonresponse bias because survey particiaption depends on satisfaction, especially after the redesign. The oberseved means rise from approximately 3.60 to 4.30 even though the true improvement is only about 0.10 points. Repsonse also falls from about 27.5% before to 15.2% after, so the better result comes from fewer and more selectively chosen respondents. A larger survey would not solve the problem because more responses collected through the same biased process would still be unrepresentative.

## Phase 2: Quantify What You Actually Know (35 pts)

### Step 2.1: Uncertainty without a formula (20 pts)

In [4]:
def bootstrap_by_hand(x, stat, B=2000, rng=rng):
    """Your own resampling loop; returns the B bootstrap statistics."""

    x = np.asarray(x)
    bootstrap_statistics = np.empty(B)

    for i in range(B):
      resample = rng.choice(x, size=len(x), replace=True)
      bootstrap_statistics[i] = stat(resample)

    return bootstrap_statistics



post_scores = after_respondents["satisfaction"].to_numpy()

# Manual bootstrap
mean_boot = bootstrap_by_hand(post_scores, np.mean)
median_boot = bootstrap_by_hand(post_scores, np.median)

manual_mean_ci = np.percentile(mean_boot, [2.5, 97.5])
manual_median_ci = np.percentile(median_boot, [2.5, 97.5])

# SciPy percentile intervals
scipy_mean_percentile = stats.bootstrap(
    (post_scores,),
    np.mean,
    confidence_level=0.95,
    n_resamples=2000,
    method="percentile",
    random_state=rng
)

scipy_median_percentile = stats.bootstrap(
    (post_scores,),
    np.median,
    confidence_level=0.95,
    n_resamples=2000,
    method="percentile",
    random_state=rng
)

# SciPy BCa intervals
scipy_mean_bca = stats.bootstrap(
    (post_scores,),
    np.mean,
    confidence_level=0.95,
    n_resamples=2000,
    method="BCa",
    random_state=rng
)

scipy_median_bca = stats.bootstrap(
    (post_scores,),
    np.median,
    confidence_level=0.95,
    n_resamples=2000,
    method="BCa",
    random_state=rng
)

print(f"Observed post mean: {np.mean(post_scores):.3f}")
print(f"Manual mean percentile CI: {manual_mean_ci}")
print(
    "SciPy mean percentile CI:",
    scipy_mean_percentile.confidence_interval
)
print(
    "SciPy mean BCa CI:",
    scipy_mean_bca.confidence_interval
)

print()

print(f"Observed post median: {np.median(post_scores):.1f}")
print(f"Manual median percentile CI: {manual_median_ci}")
print(
    "SciPy median percentile CI:",
    scipy_median_percentile.confidence_interval
)
print(
    "SciPy median BCa CI:",
    scipy_median_bca.confidence_interval
)


Observed post mean: 4.304
Manual mean percentile CI: [4.2891931  4.31920495]
SciPy mean percentile CI: ConfidenceInterval(low=np.float64(4.289979597209425), high=np.float64(4.319339871001711))
SciPy mean BCa CI: ConfidenceInterval(low=np.float64(4.289193102540477), high=np.float64(4.31881005660129))

Observed post median: 4.0
Manual median percentile CI: [4. 4.]
SciPy median percentile CI: ConfidenceInterval(low=np.float64(4.0), high=np.float64(4.0))
SciPy median BCa CI: ConfidenceInterval(low=np.float64(nan), high=np.float64(nan))


/usr/local/lib/python3.13/dist-packages/scipy/stats/_resampling.py:147: RuntimeWarning: invalid value encountered in scalar divide
  a_hat = 1/6 * sum(nums) / sum(dens)**(3/2)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:352: DegenerateDataWarning: The BCa confidence interval cannot be calculated. This problem is known to occur when the distribution is degenerate or the statistic is np.min.
  return fun(*args, **kwargs)


The manual and SciPy intervals for the post-redesign mean are almost the same, around 4.29 to 4.32, so the manual bootstrap worked. The median is 4, and its interval stays at [4, 4] because many users gave the same whole-number score. SciPy could not calculate the BCa interval for the median because the data have so many ties. These narrow intervals only measure uncertainty among the people who responded; they do not fix the survey’s response bias.

### Step 2.2: Coverage is a claim you can check (15 pts)

In [5]:
# Coverage simulation settings
M = 500
SURVEY_SIZE = 200
B_COVERAGE = 500

coverage_rng = np.random.default_rng(3916)

post_population = after["satisfaction"].to_numpy()
population_mean = post_population.mean()

percentile_hits = 0
bca_hits = 0

for i in range(M):
    survey = coverage_rng.choice(
        post_population,
        size=SURVEY_SIZE,
        replace=False
    )

    percentile_result = stats.bootstrap(
        (survey,),
        np.mean,
        confidence_level=0.95,
        n_resamples=B_COVERAGE,
        method="percentile",
        random_state=coverage_rng
    )

    bca_result = stats.bootstrap(
        (survey,),
        np.mean,
        confidence_level=0.95,
        n_resamples=B_COVERAGE,
        method="BCa",
        random_state=coverage_rng
    )

    percentile_ci = percentile_result.confidence_interval
    bca_ci = bca_result.confidence_interval

    percentile_hits += percentile_ci.low <= population_mean <= percentile_ci.high
    bca_hits += bca_ci.low <= population_mean <= bca_ci.high


percentile_coverage = percentile_hits / M
bca_coverage = bca_hits / M

percentile_mcse = np.sqrt(
    percentile_coverage * (1 - percentile_coverage) / M
)
bca_mcse = np.sqrt(
    bca_coverage * (1 - bca_coverage) / M
)

coverage_gap = abs(percentile_coverage - bca_coverage)

print(f"True post-population mean: {population_mean:.3f}")
print(
    f"Percentile coverage: {percentile_coverage:.3f} "
    f"(MC SE = {percentile_mcse:.3f})"
)
print(
    f"BCa coverage:       {bca_coverage:.3f} "
    f"(MC SE = {bca_mcse:.3f})"
)
print(f"Coverage gap:       {coverage_gap:.3f}")

True post-population mean: 3.386
Percentile coverage: 0.944 (MC SE = 0.010)
BCa coverage:       0.940 (MC SE = 0.011)
Coverage gap:       0.004


The percentile and BCa intervals both covered the true mean about 95% of the time. Their coverage rates were very close, and the difference was smaller than the Monte Carlo standard errors, so there is not enough evidence to say that one method performed better. This shows that the intervals work reasonably well with random samples, but it does not mean they can fix the biased response process from Phase 1.

## Phase 3: Test Something Worth Testing (20 pts)

### Step 3.1: The test the PM wants

In [9]:
pre_scores = before_respondents["satisfaction"].to_numpy()
post_scores = after_respondents["satisfaction"].to_numpy()

test_result = stats.ttest_ind(
    post_scores,
    pre_scores,
    equal_var = False
)

observed_difference = post_scores.mean() - pre_scores.mean()

pooled_sd = np.sqrt(
    (
      (len(pre_scores) - 1) *pre_scores.var(ddof=1)
      + (len(post_scores) - 1) * post_scores.var(ddof=1)
    )
    / (len(pre_scores) + len(post_scores) -2)
)

cohens_d = observed_difference / pooled_sd

print(f"Pre respondent mean:  {pre_scores.mean():.3f}")
print(f"Post respondent mean: {post_scores.mean():.3f}")
print(f"Observed difference:  {observed_difference:.3f}")
print(f"Welch t-statistic:    {test_result.statistic:.3f}")
print(f"P-value:              {test_result.pvalue:.3e}")
print(f"Cohen's d:            {cohens_d:.3f}")


Pre respondent mean:  3.600
Post respondent mean: 4.304
Observed difference:  0.704
Welch t-statistic:    65.166
P-value:              0.000e+00
Cohen's d:            0.864


The post-redesign survey mean is statistically significantly higher than the pre-redesign mean, with a very small p-value and a large observed effect size. However, statistical significance is not the real issue. The response process became much more selective after the redesign, so the test compares two differently biased groups. The large sample makes that biased difference look very precise, but it does not prove that the redesign caused a 0.70-point improvement.

### Step 3.2: Power, before the fact

In [7]:
from statsmodels.stats.power import TTestIndPower

power_analysis = TTestIndPower()

TRUE_EFFECT = 0.10
ALPHA = 0.05
TARGET_POWER = 0.80

standardized_effect = TRUE_EFFECT / pooled_sd

required_n = power_analysis.solve_power(
    effect_size=standardized_effect,
    alpha=ALPHA,
    power=TARGET_POWER,
    ratio=1,
    alternative="two-sided"
)

available_n = min(len(pre_scores), len(post_scores))

detectable_d = power_analysis.solve_power(
    nobs1=available_n,
    alpha=ALPHA,
    power=TARGET_POWER,
    ratio=1,
    alternative="two-sided"
)

detectable_points = detectable_d * pooled_sd

print(f"Pooled standard deviation: {pooled_sd:.3f}")
print(f"Standardized 0.10-point effect: {standardized_effect:.3f}")
print(f"Required n per group for 80% power: {np.ceil(required_n):.0f}")
print(f"Available n per group: {available_n:,}")
print(f"Detectable effect at available n: {detectable_points:.3f} points")



simulation_rng = np.random.default_rng(3916)
N_SIMULATIONS = 300
sample_sizes = [500, 2_000, 10_000]

# Recreate each post-redesign user's response probability
response_score = (
    -2.75
    + 2.00 * (after["s"] - 3.3)
    + 0.30 * after["engagement_z"]
    - 0.25 * after["had_support_ticket"]
)

post_response_probability = 1 / (1 + np.exp(-response_score))
true_post_mean = after["satisfaction"].mean()

simulation_results = []

for sample_size in sample_sizes:
    estimates = []
    respondent_counts = []

    for i in range(N_SIMULATIONS):
        selected_indices = simulation_rng.choice(
            len(after),
            size=sample_size,
            replace=False
        )

        selected_scores = after["satisfaction"].to_numpy()[selected_indices]
        selected_probabilities = post_response_probability.to_numpy()[
            selected_indices
        ]

        responded = (
            simulation_rng.uniform(size=sample_size)
            < selected_probabilities
        )

        estimates.append(selected_scores[responded].mean())
        respondent_counts.append(responded.sum())

    estimates = np.asarray(estimates)

    simulation_results.append({
        "users_contacted": sample_size,
        "average_respondents": np.mean(respondent_counts),
        "average_estimate": estimates.mean(),
        "bias": estimates.mean() - true_post_mean,
        "estimate_sd": estimates.std(ddof=1),
        "rmse": np.sqrt(np.mean((estimates - true_post_mean) ** 2))
    })

simulation_results = pd.DataFrame(simulation_results)

print(f"True post-redesign mean: {true_post_mean:.3f}")
display(simulation_results)

Pooled standard deviation: 0.815
Standardized 0.10-point effect: 0.123
Required n per group for 80% power: 1045
Available n per group: 7,597
Detectable effect at available n: 0.037 points
True post-redesign mean: 3.386


,users_contacted,average_respondents,average_estimate,bias,estimate_sd,rmse
0,500,75.930,4.298,0.912,0.081,0.916
1,2000,304.310,4.304,0.918,0.037,0.919
2,10000,1520.177,4.307,0.921,0.016,0.921


About 1,045 users per group are needed for 80% power to detect a true 0.10-point change. With the available sample, we could detect a change of about 0.04 points. However, increasing the sample does not fix the response bias. The estimates become more consistent as the sample grows, but they remain around 4.31, compared with the true mean of 3.39. More responses from the same biased process make the result more precise, not more accurate.

## Phase 4: AI Expansion — The Co-Pilot Era (15 pts)

**Status: AI usage AUTHORISED and REQUIRED.** You have built the simulation, the bootstrap and the power analysis by hand. Now use the **P.R.I.M.E. framework** to go further than you could in the time.

### Task 4.1: The weighting attempt (10 pts)

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

covariates = ["engagement_minutes", "had_support_ticket"]


def estimate_ipw_mean(data):
    """Fit the response model and return the IPW respondent mean."""

    X = data[covariates]
    response = data["responded"].astype(int)

    response_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000)
    )

    response_model.fit(X, response)

    estimated_probability = response_model.predict_proba(X)[:, 1]
    respondent_mask = response.to_numpy().astype(bool)

    respondent_scores = data["satisfaction"].to_numpy()[respondent_mask]
    respondent_probabilities = estimated_probability[respondent_mask]

    inverse_probability_weights = 1 / respondent_probabilities

    return np.average(
        respondent_scores,
        weights=inverse_probability_weights
    )


observed_mean = after.loc[
    after["responded"], "satisfaction"
].mean()

true_mean = after["satisfaction"].mean()
weighted_mean = estimate_ipw_mean(after)

print(f"Observed respondent mean: {observed_mean:.3f}")
print(f"IPW-adjusted mean:        {weighted_mean:.3f}")
print(f"True population mean:     {true_mean:.3f}")
print(f"Remaining IPW bias:       {weighted_mean - true_mean:.3f}")


# Bootstrap the complete weighting procedure
bootstrap_rng = np.random.default_rng(3916)
B = 300

bootstrap_weighted_means = np.empty(B)
population_size = len(after)

for b in range(B):
    bootstrap_indices = bootstrap_rng.integers(
        0,
        population_size,
        size=population_size
    )

    bootstrap_population = after.iloc[bootstrap_indices].copy()

    bootstrap_weighted_means[b] = estimate_ipw_mean(
        bootstrap_population
    )

bootstrap_ci = np.percentile(
    bootstrap_weighted_means,
    [2.5, 97.5]
)

print(
    "95% bootstrap interval: "
    f"[{bootstrap_ci[0]:.3f}, {bootstrap_ci[1]:.3f}]"
)

Observed respondent mean: 4.304
IPW-adjusted mean:        4.143
True population mean:     3.386
Remaining IPW bias:       0.757
95% bootstrap interval: [4.123, 4.161]


**My revision:** Version 1 only resampled respondents and kept the estimated weights fixed. In Version 2, I specified that each bootstrap sample must include the full post-redesign population and that the response model and weights must be recalculated each time. This includes uncertainty from estimating the response probabilities.

**My check:** The observed respondent mean was about **4.30**. Weighting lowered it to about **4.14**, but the known true population mean was only about **3.39**. The correction therefore reduced some bias but did not recover the truth.

**In my own words:** The logistic model estimates each user’s probability of answering based on engagement and support-ticket status. Respondents who were less likely to answer receive more weight because they represent more people like themselves. However, these weights cannot account for the direct effect of satisfaction because satisfaction is unavailable for nonrespondents.

**Would I use AI again?** Yes, I would use AI to help build and check the weighting process, but I would still verify its assumptions and results because technically correct code can rely on an invalid statistical assumption.

### Task 4.2: The slide (5 pts)

### Post-Redesign Satisfaction Survey: What We Can Conclude

Post-redesign respondents reported an average satisfaction score of approximately **4.30 out of 5**, with a 95% bootstrap interval of about **4.29 to 4.32**. This interval is narrow, so the respondent mean is estimated precisely.

However, the survey does not show that satisfaction increased by 0.70 points across all users or that the redesign caused that increase. Response fell from approximately 27% to 15%, and satisfied users became much more likely to participate. Weighting with engagement and support-ticket status reduced some bias but did not recover the known population mean because response also depended directly on satisfaction.

For future comparisons, randomly select users for the survey and actively follow up with the selected users regardless of their satisfaction. This would make the respondents more representative and allow the pre/post comparison to support a population-level conclusion.

### AI Transcript

Paste **both prompt versions exactly as you sent them** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Opus 5.5 October 26th:*

*Version 1 prompt:*

[Prep] Act as a survey statistician who works on non-response adjustment for product research.

[Request] I have a simulated user population with satisfaction scores and two covariates, engagement_minutes and had_support_ticket, plus a non-response mechanism where the probability of responding increases with satisfaction. Write Python that implements inverse-probability weighting to correct the observed survey mean, using a logistic model of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my variable names. Return the weighted estimate and its bootstrap interval.

[Mechanism Check] Explain precisely which assumption makes this correction valid, and what happens to it when response depends on satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true mean? Answer yes or no and justify it.

*Reply:*


```
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Model response using the two observed covariates
covariates = ["engagement_minutes", "had_support_ticket"]

X = after[covariates]
response = after["responded"].astype(int)

response_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

response_model.fit(X, response)

after = after.copy()
after["estimated_response_probability"] = (
    response_model.predict_proba(X)[:, 1]
)

# Calculate the IPW-adjusted respondent mean
respondents = after.loc[after["responded"]].copy()

respondents["ipw"] = (
    1 / respondents["estimated_response_probability"]
)

weighted_mean_v1 = np.average(
    respondents["satisfaction"],
    weights=respondents["ipw"]
)

observed_mean_v1 = respondents["satisfaction"].mean()

# Version 1 bootstrap:
# resample respondents while keeping their estimated weights fixed
bootstrap_rng_v1 = np.random.default_rng(3916)
B_V1 = 2000
bootstrap_weighted_means_v1 = np.empty(B_V1)

scores_v1 = respondents["satisfaction"].to_numpy()
weights_v1 = respondents["ipw"].to_numpy()
n_respondents = len(respondents)

for b in range(B_V1):
    indices = bootstrap_rng_v1.integers(
        0,
        n_respondents,
        size=n_respondents
    )

    bootstrap_weighted_means_v1[b] = np.average(
        scores_v1[indices],
        weights=weights_v1[indices]
    )

bootstrap_ci_v1 = np.percentile(
    bootstrap_weighted_means_v1,
    [2.5, 97.5]
)

print(f"Observed respondent mean: {observed_mean_v1:.3f}")
print(f"Version 1 IPW mean:       {weighted_mean_v1:.3f}")
print(
    "Version 1 95% interval: "
    f"[{bootstrap_ci_v1[0]:.3f}, {bootstrap_ci_v1[1]:.3f}]"
)
```




*Version 2 prompt:*

[Prep] Act as a survey statistician who works on non-response adjustment for product research.

[Request] I have a simulated post-redesign user population in a DataFrame named after. It contains satisfaction, engagement_minutes, had_support_ticket, and responded. Write Python that uses inverse-probability weighting to estimate the post-redesign population mean satisfaction. Fit a logistic response model using engagement_minutes and had_support_ticket.

[Iterate] Use numpy, pandas, and scikit-learn. Reuse my exact variable and column names. For the bootstrap, resample rows from the full post-redesign population, refit the logistic response model in every resample, recalculate the inverse-probability weights, and then calculate the weighted respondent mean. Return the weighted estimate and a 95% percentile bootstrap interval.

[Mechanism Check] Explain precisely which assumption makes this correction valid and what happens when response depends directly on satisfaction after controlling for the covariates.

[Evaluate] Compare the weighted estimate with the known true post-redesign population mean from after["satisfaction"].mean(). State whether the correction worked and explain why.

*Reply:*



```
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

covariates = ["engagement_minutes", "had_support_ticket"]


def estimate_ipw_mean(data):
    """Fit the response model and return the IPW respondent mean."""

    X = data[covariates]
    response = data["responded"].astype(int)

    response_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000)
    )

    response_model.fit(X, response)

    estimated_probability = (
        response_model.predict_proba(X)[:, 1]
    )

    respondent_mask = response.to_numpy().astype(bool)

    respondent_scores = (
        data["satisfaction"].to_numpy()[respondent_mask]
    )

    respondent_probabilities = (
        estimated_probability[respondent_mask]
    )

    inverse_probability_weights = (
        1 / respondent_probabilities
    )

    return np.average(
        respondent_scores,
        weights=inverse_probability_weights
    )


# Estimate the observed, weighted, and true means
observed_mean_v2 = after.loc[
    after["responded"], "satisfaction"
].mean()

true_mean_v2 = after["satisfaction"].mean()
weighted_mean_v2 = estimate_ipw_mean(after)

# Version 2 bootstrap:
# resample the full population and refit the model each time
bootstrap_rng_v2 = np.random.default_rng(3916)
B_V2 = 300
population_size = len(after)

bootstrap_weighted_means_v2 = np.empty(B_V2)

for b in range(B_V2):
    bootstrap_indices = bootstrap_rng_v2.integers(
        0,
        population_size,
        size=population_size
    )

    bootstrap_population = after.iloc[
        bootstrap_indices
    ].copy()

    bootstrap_weighted_means_v2[b] = estimate_ipw_mean(
        bootstrap_population
    )

bootstrap_ci_v2 = np.percentile(
    bootstrap_weighted_means_v2,
    [2.5, 97.5]
)

print(f"Observed respondent mean: {observed_mean_v2:.3f}")
print(f"Version 2 IPW mean:       {weighted_mean_v2:.3f}")
print(f"True population mean:     {true_mean_v2:.3f}")
print(
    f"Remaining IPW bias:       "
    f"{weighted_mean_v2 - true_mean_v2:.3f}"
)
print(
    "Version 2 95% interval: "
    f"[{bootstrap_ci_v2[0]:.3f}, {bootstrap_ci_v2[1]:.3f}]"
)
```

